# 00 · Setup Check

Run this notebook before starting any lab. It validates your Python environment
and reports which of the 4 cases (Docker profiles) are currently reachable, so
you always know what you're ready to run.

This notebook is safe to run at any time, with any subset of `make up-*`
profiles running (or none at all — Case A needs no Docker).

In [ ]:
import sys

print(f"Python: {sys.version}")
assert sys.version_info >= (3, 12), "This lab targets Python 3.12+"
print("✅ Python version OK")

In [ ]:
import pyspark

print(f"PySpark: {pyspark.__version__}")
assert pyspark.__version__.startswith("3.5"), "Expected PySpark 3.5.x"
print("✅ PySpark version OK")

In [ ]:
# Case A needs no Docker at all — local[*] ships inside the pyspark pip package.
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_local_session

spark = get_local_session("00-setup-check")
df = spark.range(5).toDF("n")
assert df.count() == 5
print("✅ Local Spark session works (Case A ready)")
spark.stop()

## Which Docker profiles are up?

The cell below probes the ports each profile exposes. It won't start anything
— it just tells you what's already running.

In [ ]:
import socket


def port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


checks = {
    "Case B/D — Spark Connect (localhost:15002)": ("localhost", 15002),
    "Case B — Spark Master UI (localhost:8080)": ("localhost", 8080),
    "Case D — RustFS S3 API (localhost:9000)": ("localhost", 9000),
    "Case C — YARN ResourceManager (localhost:8088)": ("localhost", 8088),
    "Case C — HDFS HttpFS gateway (localhost:14000)": ("localhost", 14000),
}

for label, (host, port) in checks.items():
    status = "✅ up" if port_open(host, port) else "⬜ not running"
    print(f"{status:12s} {label}")

## Dataset check

Cases B/C/D expect a dataset generated via `make generate-data`. Run the cell
below to confirm it exists (or generate the `small` scale right now).

In [ ]:
import subprocess
from pathlib import Path

data_dir = Path("../data/bronze")
if data_dir.exists() and any(data_dir.iterdir()):
    print(f"✅ Dataset found at {data_dir.resolve()}")
    for table in ["empresas", "funcionarios", "vendas"]:
        print(f"   - {table}: {'present' if (data_dir / table).exists() else 'MISSING'}")
else:
    print("⬜ No dataset found. Generating scale=small now...")
    subprocess.run(
        ["uv", "run", "python", "scripts/generate_dataset.py", "--scale", "small"],
        cwd="..",
        check=True,
    )